# 🛡️ SentinelNLP: Fine-tune XLM-RoBERTa on Multilingual CVEs (Free T4 GPU)

This notebook trains `xlm-roberta-base` for cross-lingual vulnerability severity classification across **English, Hindi, Chinese, Spanish, Arabic, and French**.

### Steps:
1. Switch runtime to GPU (**Runtime → Change runtime type → T4 GPU**)
2. Upload your `nvd_records.json` from `data/processed/nvd_records.json`
3. Fine-tune the model with mixed precision (fp16)
4. Download the `models_xlmr.zip` bundle and extract into `MAJOR/models/xlmr`

In [ ]:
!pip install -q transformers datasets accelerate torch evaluate scikit-learn

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️ Please enable GPU under Runtime -> Change runtime type!')

### 1. Upload `nvd_records.json`
Upload the file from your local computer: `c:\Users\abhik\Desktop\MAJOR\data\processed\nvd_records.json`

In [ ]:
from google.colab import files
import os

if not os.path.exists('nvd_records.json'):
    print('Please upload nvd_records.json:')
    uploaded = files.upload()
else:
    print('nvd_records.json already exists!')

### 2. Dataset Preparation & Stratified Splitting

In [ ]:
import json
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

with open('nvd_records.json', 'r', encoding='utf-8') as f:
    records = json.load(f)

print(f'Total records loaded: {len(records)}')

texts = [r['description'] for r in records]
labels = [r['severity'] for r in records]

unique_labels = sorted(list(set(labels)))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}
print('Labels:', label2id)

# Split into train & test (80/20 stratified)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

MODEL_NAME = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=256, padding='max_length')

train_ds = Dataset.from_dict({'text': train_texts, 'label': [label2id[l] for l in train_labels]})
test_ds = Dataset.from_dict({'text': test_texts, 'label': [label2id[l] for l in test_labels]})

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

### 3. Initialize XLM-RoBERTa Classifier & Metrics

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'macro_f1': f1}

### 4. GPU Fine-Tuning Execution

In [ ]:
OUTPUT_DIR = './models_xlmr'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print('🚀 Starting fine-tuning...')
trainer.train()

### 5. Export Model & Download Zip

In [ ]:
final_model_path = './xlmr_exported'
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

with open(f'{final_model_path}/label_mapping.json', 'w') as f:
    json.dump(label2id, f, indent=2)

!zip -r models_xlmr.zip xlmr_exported
files.download('models_xlmr.zip')
print('✅ Finished! Extract the contents into MAJOR/models/xlmr.')